In [ ]:
import os
import pandas as pd
import numpy as np

from quick_pp.objects import Project

####  Steps to create project
1. Run the next cell.
2. Specify the example either clastic or carbonate.
3. NNS_2-5.qppp project will be saved in data\04_project folder.

* Note that the required curves in the LAS files are 'GR', 'RT', 'NPHI', 'RHOB'

In [ ]:
data_path = 'data/'
filenames = []
for root, dirs, files in os.walk(data_path):
    for file in files:
        filenames.append(os.path.join(root, file)) if file.endswith('.las') and not 'seismic' in root  else None
print(filenames)

project_name = "NNS_2-5"
project = Project(name=project_name)
project.read_las(filenames)
# clear_output()               

In [ ]:
# Clean up data
data = project.get_all_data()
remove_rows = (
    (data.WELL_NAME == '15-9-19-SR') & ((data.DEPTH < 560) | (data.DEPTH > 4590))  # Placeholder
)
merged_df = data[~remove_rows].reset_index(drop=True)

# Convert DEPTH from feet to m
merged_df['DEPTH'] = round(merged_df.DEPTH / 3.281, 4)
merged_df['NPHI'] = merged_df.NPHI / 100
merged_df['GR'] = np.where(merged_df.GRS.notna(), merged_df.GRS, merged_df.GRD)
merged_df['RT'] = merged_df.ILD
merged_df['DTC'] = merged_df.DT
merged_df['CALI'] = np.where(merged_df.CALI.notna(), merged_df.CALI, merged_df.CALS)

In [ ]:
# Process core data
all_core_data = pd.read_excel(r"data\CoreAnalysis_Export_Selected_GBR_2_05.xls")
all_core_data['WELL_NAME'] = all_core_data['Wellname'].str.replace('/0','-').str.replace('/','-').str.replace(' ', '')
all_core_data['DEPTH'] = pd.to_numeric(all_core_data['Sample_Depth'], errors='coerce')
all_core_data['DEPTH'] = round(all_core_data['DEPTH'] / 3.281, 4)  # Convert to meter
all_core_data['CPORE'] = pd.to_numeric(all_core_data['Porosity_1'], errors='coerce') / 100
all_core_data['CPERM'] = pd.to_numeric(all_core_data['Hor_Permeability_1'], errors='coerce')

merged_core_df = pd.DataFrame()
return_df = pd.DataFrame()
for well_name, well_data in merged_df.groupby('WELL_NAME'):
    temp_df = well_data.copy()
    core_data = all_core_data[all_core_data.WELL_NAME == well_name]
    core_data = core_data.sort_values('DEPTH')
    num_core_points = len(core_data)
    core_ids = [str(i) for i in range(1, num_core_points + 1)]
    core_data['CORE_ID'] = core_ids
    merged_core_df = pd.concat([merged_core_df, core_data])

    # Merge with well logs
    temp_df = pd.merge_asof(well_data, core_data[['DEPTH', 'CPORE', 'CPERM', 'CORE_ID']],
                            direction='nearest', on='DEPTH', tolerance=0.1524 / 2)
    return_df = pd.concat([return_df, temp_df], ignore_index=True)
merged_core_df['WELL_NAME'] = merged_core_df['WELL_NAME'].astype(str)
merged_core_df.to_csv(r'data\merged_core_data.csv', index=False)
merged_df = return_df.copy()

In [ ]:
# Process zones
tops_df = pd.read_excel(r"data\NNS 2-5_Tops.xlsx")
tops_df['WELL_NAME'] = tops_df['Wellbore name'].str.replace('/0','-').str.replace('/','-').str.replace(' ', '')
return_df = pd.DataFrame()
for well_name, well_data in merged_df.groupby('WELL_NAME'):
    marker_df = tops_df[(tops_df.WELL_NAME == well_name) & (tops_df.Level == 'FORMATION')].copy()
    marker_df['DEPTH'] = marker_df['Top depth [m]'].astype('float')
    marker_df['ZONES'] = marker_df['Lithostrat. unit']

    # Merge with well logs
    temp_df = pd.merge_asof(well_data, marker_df[['DEPTH', 'ZONES']], direction='backward', on='DEPTH')
    temp_df = temp_df.sort_values('DEPTH').reset_index(drop=True)
    return_df = pd.concat([return_df, temp_df], ignore_index=True)
merged_df = return_df.copy()

In [ ]:
# Process TVDSS
import wellpathpy as wpp

filenames = os.listdir(r'data')
return_df = pd.DataFrame()
for well_name, well_data in merged_df.groupby('WELL_NAME'):
    filename = [f for f in filenames if well_name in f and 'Deviation' in f]
    
    temp_df = well_data.drop_duplicates('DEPTH')
    if filename:
        filepath = os.path.join('data', filename[0])
        print(filepath)
        data_dict = pd.read_excel(filepath, sheet_name=None)
        survey_key = [k for k in data_dict.keys() if 'Dev' in k][0]
        well_survey = data_dict.get(survey_key)
        well_survey = well_survey.rename(columns={
            'Well Name': 'WELL_NAME', 'MD M': 'md', 'Inclination': 'incl', 'Azimuth': 'azim',
            'Grid_Easting': 'x', 'Grid_Northing': 'y', 'TVD M': 'TVD', 'TVDSS M': 'TVDSS'
            })
        well_survey['WELL_NAME'] = well_survey['WELL_NAME'].str.replace('/0','-').str.replace('/','-').str.replace(' ', '')
        coords = well_survey[well_survey.WELL_NAME == well_name]
        dev_survey = wpp.deviation(coords['md'], coords['incl'], coords['azim'])
        tvd = dev_survey.minimum_curvature().resample(temp_df.DEPTH.values).depth
        if len(tvd) == len(temp_df):
            temp_df['TVD'] = tvd
    return_df = pd.concat([return_df, temp_df])

In [ ]:
from quick_pp.plotter.plotter import plotly_log

# Plot the results
well_name = '2-5-1'
well_data = merged_df[merged_df.WELL_NAME == well_name]
fig = plotly_log(well_data, well_name=well_name, depth_uom='m')
fig.show(config=dict(scrollZoom=True))

In [ ]:
from quick_pp.plotter.plotter import plotly_log

# Plot the results
folder = r'data\04_project\outputs'
for well_name, well_data in merged_df.groupby('WELL_NAME'):
    well_data = merged_df[merged_df.WELL_NAME == well_name]
    fig = plotly_log(well_data, well_name=well_name, depth_uom='m')
    fig.write_html(os.path.join(folder, f'{well_name}_plot.html'), config=dict(scrollZoom=True))

In [ ]:
# Save the update data to the project
project.update_data(merged_df)
project.save()